## Invoice DocAI (v2) — 03b Donut Fine-tuning on SROIE

**Goal:** Fine-tune `naver-clova-ix/donut-base` on SROIE train set (626 docs)
to produce structured output: `<s_company>`, `<s_date>`, `<s_total>`.

**Why:** Pre-trained checkpoints (`philschmid/donut-base-sroie`, CORD) produce
poor/unstructured output for SROIE invoice fields. Fine-tuning is required.

In [ ]:
import sys
from pathlib import Path

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

def is_project_root(p: Path) -> bool:
    return (p / 'data' / 'sroie').exists() and ((p / 'notebooks').exists() or (p / 'v2' / 'notebooks').exists())

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd, cwd.parent, cwd.parent.parent,
        cwd / 'invoice_docai',
        Path('/content/drive/MyDrive/ORC/invoice_docai'),
        Path('/content/drive/MyDrive/invoice_docai'),
        Path('/content/drive/MyDrive/From OCR 2 Transformers/invoice_docai'),
        Path(r'c:\Yandex.Disk\Yandex.Disk\ML Neimark\From OCR 2 Transformers\invoice_docai'),
    ]
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if is_project_root(p):
            return p
    raise FileNotFoundError('Cannot locate project root')

PROJECT_ROOT = resolve_project_root()
V2_SRC = PROJECT_ROOT / 'v2' / 'src'
if V2_SRC.exists() and str(V2_SRC) not in sys.path:
    sys.path.insert(0, str(V2_SRC))
elif (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from docai_utils import normalize_date, normalize_total

PROCESSED = PROJECT_ROOT / 'data' / 'sroie' / 'processed'
CHECKPOINT_DIR = PROJECT_ROOT / 'v2' / 'checkpoints' / 'donut-sroie-finetuned'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = PROJECT_ROOT / 'v2' / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT   = {PROJECT_ROOT}')
print(f'CHECKPOINT_DIR = {CHECKPOINT_DIR}')

In [ ]:
# Install deps if needed
try:
    from transformers import DonutProcessor, VisionEncoderDecoderModel
    print('transformers OK')
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'transformers', 'sentencepiece', 'accelerate', '-q'])
    from transformers import DonutProcessor, VisionEncoderDecoderModel

import json, re, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

## 1. Load data

In [ ]:
manifest_train = pd.read_csv(PROCESSED / 'manifest_train.csv').dropna(subset=['image_path']).reset_index(drop=True)
manifest_val   = pd.read_csv(PROCESSED / 'manifest_val.csv').dropna(subset=['image_path']).reset_index(drop=True)

print(f'Train: {len(manifest_train)}, Val: {len(manifest_val)}')
manifest_train.head(2)

## 2. Load base model + add special tokens

In [ ]:
BASE_MODEL = 'naver-clova-ix/donut-base'

processor = DonutProcessor.from_pretrained(BASE_MODEL)
model     = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)

# Special tokens for SROIE task
SPECIAL_TOKENS = [
    '<s_sroie>', '</s_sroie>',
    '<s_company>', '</s_company>',
    '<s_date>', '</s_date>',
    '<s_total>', '</s_total>',
    '<s_address>', '</s_address>',
]

newly_added = processor.tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Set decoder start token
TASK_PROMPT = '<s_sroie>'
task_prompt_id = processor.tokenizer.convert_tokens_to_ids(TASK_PROMPT)
model.config.decoder_start_token_id = task_prompt_id
model.config.pad_token_id = processor.tokenizer.pad_token_id

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Added {newly_added} special tokens')
print(f'Vocab size: {len(processor.tokenizer)}')

## 3. Dataset class

In [ ]:
class SROIEDonutDataset(Dataset):
    """SROIE dataset for Donut fine-tuning."""

    def __init__(self, manifest_df, processor, max_length=256):
        self.manifest = manifest_df.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        pixel_values = self.processor(image, return_tensors='pt').pixel_values.squeeze(0)

        vendor = str(row.get('gt_vendor', '') or '')
        date   = str(row.get('gt_date', '') or '')
        total  = str(row.get('gt_total', '') or '')

        target = (
            f'<s_sroie>'
            f'<s_company>{vendor}</s_company>'
            f'<s_date>{date}</s_date>'
            f'<s_total>{total}</s_total>'
            f'</s_sroie>'
        )

        encoding = self.processor.tokenizer(
            target,
            add_special_tokens=False,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        labels = encoding.input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


train_ds = SROIEDonutDataset(manifest_train, processor)
val_ds   = SROIEDonutDataset(manifest_val, processor)

print(f'Train dataset: {len(train_ds)}')
print(f'Val dataset:   {len(val_ds)}')

# Sanity check
sample = train_ds[0]
print(f'pixel_values shape: {sample["pixel_values"].shape}')
print(f'labels shape: {sample["labels"].shape}')
print(f'labels (non-pad): {(sample["labels"] != -100).sum().item()} tokens')

## 4. Training loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR

# Hyperparameters
EPOCHS     = 5
BATCH_SIZE = 2       # Colab T4 has ~16 GB VRAM; 2 is safe
LR         = 5e-5
WARMUP_FRAC = 0.1
LOG_EVERY  = 50

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

model = model.to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps)

print(f'Total steps: {total_steps}, Warmup: {warmup_steps}')
print(f'Batches/epoch: {len(train_loader)}')

In [ ]:
best_loss = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for step, batch in enumerate(pbar, 1):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        n_batches += 1

        if step % LOG_EVERY == 0:
            avg = epoch_loss / n_batches
            pbar.set_postfix(loss=f'{avg:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    avg_loss = epoch_loss / max(n_batches, 1)
    history.append({'epoch': epoch, 'train_loss': avg_loss})
    print(f'  Epoch {epoch} — avg loss: {avg_loss:.4f}')

    # Save best checkpoint
    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save_pretrained(CHECKPOINT_DIR)
        processor.save_pretrained(CHECKPOINT_DIR)
        print(f'  -> Saved best checkpoint (loss={best_loss:.4f})')

print(f'\nTraining complete. Best loss: {best_loss:.4f}')

In [ ]:
# Plot training curve
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
plt.figure(figsize=(8, 4))
plt.plot(hist_df['epoch'], hist_df['train_loss'], 'o-', label='Train loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Donut SROIE Fine-tuning')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Validate fine-tuned model

In [ ]:
# Load best checkpoint
ft_processor = DonutProcessor.from_pretrained(CHECKPOINT_DIR)
ft_model     = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT_DIR).to(device)
ft_model.eval()

TASK_PROMPT = '<s_sroie>'

def donut_predict_ft(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = ft_processor(image, return_tensors='pt').pixel_values.to(device)

    decoder_input_ids = ft_processor.tokenizer(
        TASK_PROMPT, add_special_tokens=False, return_tensors='pt'
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = ft_model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=ft_model.decoder.config.max_position_embeddings,
            pad_token_id=ft_processor.tokenizer.pad_token_id,
            eos_token_id=ft_processor.tokenizer.eos_token_id,
            num_beams=1,
            use_cache=True,
            bad_words_ids=[[ft_processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    seq = ft_processor.batch_decode(outputs.sequences)[0]
    seq = seq.replace(ft_processor.tokenizer.eos_token, '').replace(ft_processor.tokenizer.pad_token, '')

    # Parse SROIE tags
    vendor = ''
    date = ''
    total = ''

    m = re.search(r'<s_company>(.*?)</s_company>', seq, re.DOTALL)
    if m:
        vendor = m.group(1).strip()
    m = re.search(r'<s_date>(.*?)</s_date>', seq, re.DOTALL)
    if m:
        date = normalize_date(m.group(1).strip())
    m = re.search(r'<s_total>(.*?)</s_total>', seq, re.DOTALL)
    if m:
        total = normalize_total(m.group(1).strip())

    return {'pred_vendor': vendor, 'pred_date': date, 'pred_total': total, 'raw_text': seq}

print('Fine-tuned model loaded. Running validation...')

In [ ]:
# Quick validation on 50 docs
from docai_utils import evaluate

N_VAL = min(50, len(manifest_val))
val_subset = manifest_val.sample(N_VAL, random_state=42).reset_index(drop=True)

ft_rows = []
ft_latencies = []

for _, row in tqdm(val_subset.iterrows(), total=len(val_subset), desc='FT validation'):
    t0 = time.perf_counter()
    pred = donut_predict_ft(row['image_path'])
    elapsed = (time.perf_counter() - t0) * 1000
    ft_rows.append({'id': row['id'], **{k: v for k, v in pred.items() if k != 'raw_text'}})
    ft_latencies.append(elapsed)

ft_pred_df = pd.DataFrame(ft_rows)
ft_metrics, ft_errors = evaluate(val_subset[['id', 'gt_vendor', 'gt_date', 'gt_total']], ft_pred_df)

print(f'\n=== Fine-tuned Donut Metrics (N={N_VAL}) ===')
display(ft_metrics)
print(f'Latency mean: {np.mean(ft_latencies):.0f} ms/doc')

## 6. Full validation (all 347 docs) & save predictions

In [ ]:
# Full run on entire val set
all_rows = []
all_latencies = []

checkpoint_path = OUTPUT_ROOT / 'donut_ft_checkpoint.csv'
done_ids = set()
if checkpoint_path.exists():
    done_df = pd.read_csv(checkpoint_path)
    done_ids = set(done_df['id'].tolist())
    all_rows = done_df.to_dict('records')
    print(f'Resuming from checkpoint: {len(done_ids)} already done')

for _, row in tqdm(manifest_val.iterrows(), total=len(manifest_val), desc='Full FT inference'):
    if row['id'] in done_ids:
        continue
    t0 = time.perf_counter()
    pred = donut_predict_ft(row['image_path'])
    elapsed = (time.perf_counter() - t0) * 1000
    all_rows.append({
        'id': row['id'],
        'pred_vendor': pred['pred_vendor'],
        'pred_date': pred['pred_date'],
        'pred_total': pred['pred_total'],
    })
    all_latencies.append(elapsed)

    # Checkpoint every 20 docs
    if len(all_rows) % 20 == 0:
        pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False)

full_pred_df = pd.DataFrame(all_rows)
full_pred_df.to_csv(OUTPUT_ROOT / 'donut_predictions_clean.csv', index=False)
if checkpoint_path.exists():
    checkpoint_path.unlink()

full_metrics, _ = evaluate(manifest_val[['id', 'gt_vendor', 'gt_date', 'gt_total']], full_pred_df)
print(f'\n=== Fine-tuned Donut — Full Val Metrics ===')
display(full_metrics)
print(f'Saved: {OUTPUT_ROOT / "donut_predictions_clean.csv"}')
print(f'Latency mean: {np.mean(all_latencies):.0f} ms/doc' if all_latencies else 'Latency: resumed from checkpoint')

## Done

Fine-tuned model saved to `v2/checkpoints/donut-sroie-finetuned/`.
Predictions saved to `v2/outputs/donut_predictions_clean.csv`.

Next: run 04_robustness_eval.ipynb to test both pipelines on corrupted images.